In [16]:
import os, json, time

In [13]:
from together import Together
import utils

key_file = 'together-personal.txt'
with open(key_file, 'r') as f:
    API_KEY = f.read().strip()

client = Together(
  api_key=API_KEY
)


In [ ]:
from importlib import reload
reload(utils)

In [10]:
model_name = 'llama3-turbo'
model_endpoint = utils.model_names_to_endpoints[model_name]
model_endpoint

'meta-llama/Llama-3.3-70B-Instruct-Turbo'

In [12]:
data_dir = '../data/final_dataset'
short_ans_files = ['certamen_short_answer.json', 'junior_scholarship_short_answer.json']
short_ans_files = [os.path.join(data_dir, f) for f in short_ans_files]

file_to_data = {}
for file in short_ans_files:
    base_name = os.path.basename(file)
    file_to_data[base_name] = []
    with open(file, 'r') as f:
        file_to_data[base_name] += json.load(f)

    print(base_name, len(file_to_data[base_name]))

certamen_short_answer.json 4596
junior_scholarship_short_answer.json 675


In [8]:
def construct_short_ans_one_word_user_prompt(q_dict):
    question_text = q_dict['question'] if 'question' in q_dict else q_dict['question_text']
    question_text += '\n' + utils.short_ans_one_word_format_instructions

    return question_text

In [9]:
prompt = construct_short_ans_one_word_user_prompt(file_to_data['certamen_short_answer.json'][0])
prompt

'What name was given to the large agricultural estates which resulted from the distribution of the ager publicus in the 2nd century B.C.?\nAt the end of your response, give your answer as a single word like this:\nAnswer: Word'

In [14]:
response = client.chat.completions.create(
  model=model_endpoint,
  messages=[
    {
        "role": "system",
        "content": utils.sys_prompt
    },
    {
      "role": "user",
      "content": prompt
    }
  ],
  temperature=0.6, 
  top_p=0.95, 
  #min_p=0,
  #top_k=20
)
print(response.choices[0].message.content)

The large agricultural estates that emerged as a result of the distribution of the ager publicus in the 2nd century B.C. were known as latifundia. These estates were characterized by their large size and the use of slave labor, which led to the displacement of small-scale farmers and contributed to social and economic changes in ancient Rome.

Answer: Latifundia


In [15]:
save_dir = f'../data/model_responses/{model_name}'
if not os.path.exists(save_dir):
    os.makedirs(save_dir)

In [ ]:
for filename, data in file_to_data.items():
    print(filename)
    q_id_to_resp = {}
    i = 0
    save_file = os.path.join(save_dir, filename)
    for q_dict in data:
        q_id = q_dict['question_id']
        prompt = construct_short_ans_one_word_user_prompt(q_dict)

        response = client.chat.completions.create(
            model=model_endpoint,
            messages=[
                {
                    "role": "system",
                    "content": utils.sys_prompt
                },
                {
                "role": "user",
                "content": prompt
                }
            ],
            temperature=0.6, 
            top_p=0.95, 
            #min_p=0,
            #top_k=20
        )
        try:
            resp = response.choices[0].message.content
        except:
            print(f'Error on {q_id}')
            print(response)
            # dump this file 
            with open(save_file, 'w') as f:
                json.dump(q_id_to_resp, f, indent=4)
            break
        q_id_to_resp[q_id] = resp

        time.sleep(.1)
        

        if i % 100 == 0:
            # dump this file 
            print(f'  {i} / {len(data)}')
            with open(save_file, 'w') as f:
                json.dump(q_id_to_resp, f, indent=4)
            
        i += 1

    # dump this file 
    with open(save_file, 'w') as f:
        json.dump(q_id_to_resp, f, indent=4)